In [ ]:
import pandas as pd

# Authoritative catalog (129 galaxies) -- rar_with_ml_direct_phot_filtered.csv
# (81 galaxies) is superseded, confirmed 2026-08-24 (see README.md).
# sep=None + engine="python" auto-sniffs the delimiter: this file is
# whitespace-separated, unlike the old comma-separated CSV.
df = pd.read_csv("ALL_gbar_gobs_RAR_direct_phot.txt", sep=None, engine="python")

# Ver las primeras filas
display(df.head())

In [ ]:
galaxy_names = df["Galaxy"].unique()

print("Unique galaxy names:", len(galaxy_names))

#do a hist of the number of data points per galaxy
import matplotlib.pyplot as plt
galaxy_counts = df["Galaxy"].value_counts()
plt.figure(figsize=(10, 6))
galaxy_counts.plot(kind="bar")
plt.xlabel("Galaxy")
plt.ylabel("Number of Data Points")
plt.title("Number of Data Points per Galaxy")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#do a hist
plt.figure(figsize=(10, 6))
plt.hist(galaxy_counts, bins=20, edgecolor="black")
plt.xlabel("Rar_with_ML_direct_phot")
plt.ylabel("Frequency")
plt.title("Distribution of Rar_with_ML_direct_phot")
plt.tight_layout()
plt.show()

#median and mean of number data points per galaxy
median_count = galaxy_counts.median()
mean_count = galaxy_counts.mean() 
print(f"Median number of data points per galaxy: {median_count}")
print(f"Mean number of data points per galaxy: {mean_count}")



In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

with PdfPages('all_velocity_profiles.pdf') as pdf:

    for galaxy in galaxy_names:
        galaxy_data = df[df["Galaxy"] == galaxy]

        r = galaxy_data["Radius_kpc"]
        vrot = galaxy_data["v_rot"]
        vrot_err_lo = galaxy_data["v_rot_err_lo"]
        vrot_err_hi = galaxy_data["v_rot_err_hi"]
        vbar = galaxy_data["v_bar_total"]
        vbar_err = galaxy_data["v_bar_total_err"]

        fig, ax = plt.subplots(figsize=(10, 6))

        ax.errorbar(
            r, vrot,
            yerr=[vrot_err_lo, vrot_err_hi],
            fmt='o',
            label='v_rot',
            capsize=5
        )

        ax.errorbar(
            r, vbar,
            yerr=vbar_err,
            fmt='o',
            label='v_bar_total',
            capsize=5
        )

        ax.set_xlabel('Radius (kpc)')
        ax.set_ylabel('Velocity (km/s)')
        ax.set_title(f'Galaxy: {galaxy}')
        ax.legend()

        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print("Saved to all_velocity_profiles.pdf")

#plot galaxies with the most data points
top_galaxies = galaxy_counts.head(5).index
top_galaxy_data = df[df["Galaxy"].isin(top_galaxies)]
plt.figure(figsize=(10, 6))
for galaxy in top_galaxies:
    galaxy_data = top_galaxy_data[top_galaxy_data["Galaxy"] == galaxy]
    print(f"Plotting galaxy: {galaxy} with {len(galaxy_data)} data points")
    plt.errorbar(
        galaxy_data["Radius_kpc"],
        galaxy_data["v_rot"],
        yerr=[galaxy_data["v_rot_err_lo"], galaxy_data["v_rot_err_hi"]],
        fmt='o',
        label=galaxy,
        capsize=5
    )
    # plt.errorbar(
    #     galaxy_data["Radius_kpc"],
    #     galaxy_data["v_bar_total"],
    #     yerr=galaxy_data["v_bar_total_err"],
    #     fmt='o',
    #     label=f'{galaxy} v_bar_total',
    #     capsize=5
    # )

plt.xlabel('Radius (kpc)')
plt.ylabel('Velocity (km/s)')
plt.title('Velocity Profiles of Top 5 Galaxies')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#do a hist of the snr of the v_rot measurements
plt.figure(figsize=(10, 6))
snr_v_rot = df["v_rot"] / ((df["v_rot_err_lo"] + df["v_rot_err_hi"]) / 2)
plt.hist(snr_v_rot, bins=20, edgecolor="black")
plt.xlabel("SNR of v_rot")
plt.ylabel("Frequency")
plt.title("Distribution of SNR for v_rot Measurements")
plt.tight_layout()
plt.show()

#find galaxy with an snr of 160 and plot it
galaxy_snr_160 = df[snr_v_rot > 140]["Galaxy"].iloc[0]
print("Galaxy with SNR of 160 for v_rot:", galaxy_snr_160)
galaxy_data = df[df["Galaxy"] == galaxy_snr_160]
plt.figure(figsize=(10, 6))
plt.errorbar(
    galaxy_data["Radius_kpc"],
    galaxy_data["v_rot"],
    yerr=[galaxy_data["v_rot_err_lo"], galaxy_data["v_rot_err_hi"]],
    fmt='o',
    label='v_rot',
    capsize=5
)
plt.errorbar(
    galaxy_data["Radius_kpc"],
    galaxy_data["v_bar_total"],
    yerr=galaxy_data["v_bar_total_err"],
    fmt='o',
    label='v_bar_total',
    capsize=5
)
plt.xlabel('Radius (kpc)')
plt.ylabel('Velocity (km/s)')
plt.title(f'Galaxy: {galaxy_snr_160} with SNR of 160 for v_rot')
plt.legend()
plt.tight_layout()
plt.show()